# Consolidação da referência Pareto 8D

Une as seis populações finais de NSGA-III/MOEA-D e a referência legada previamente auditada. Reavalia todas as soluções pelo RSM, remove inviáveis, duplicatas e dominadas e exporta a nova referência com rastreabilidade.

In [ ]:
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting

def project_root(start=Path.cwd()):
    for p in (start.resolve(), *start.resolve().parents):
        if (p/'AGENTS.md').exists() and (p/'notebooks').exists(): return p
    raise FileNotFoundError('Raiz do projeto não encontrada.')
ROOT=project_root(); OUT=ROOT/'results'/'applied'/'reference_8d'; RUNS=OUT/'runs'; FINAL=ROOT/'data'/'reference_fronts'; FINAL.mkdir(parents=True,exist_ok=True)
cfg=json.loads((OUT/'reference_run_config.json').read_text(encoding='utf-8')); ALPHA=float(cfg['alpha']); config_hash=cfg['config_hash']
xs=['cs','f','md']; ys=['T','MTTF','WR','Ra','Rt','Kp','ROI','OEE']; signs=np.array([-1,-1,1,1,1,1,-1,-1.])
excel=Path(cfg['excel']); d=pd.read_excel(excel); d.columns=[str(c).strip() for c in d.columns]
model=Pipeline([('poly',PolynomialFeatures(2,include_bias=False)),('reg',LinearRegression())]).fit(d[xs].to_numpy(float),d[ys].to_numpy(float))
sources=[]
for method in ('NSGAIII','MOEAD'):
    for seed in (1010,2110,3070):
        p=RUNS/f'{method}_seed{seed}_final.npz'
        if not p.exists(): raise FileNotFoundError(f'Execução ainda incompleta: {p}')
        z=np.load(p,allow_pickle=False)
        if str(z['config_hash'])!=config_hash: raise RuntimeError(f'Configuração incompatível: {p}')
        sources.append((f'{method}_seed{seed}',np.asarray(z['X'],float)))
legacy=Path.home()/'Documents'/'Dissertação'/'04_CODIGOS'/'notebooks'/'files'/'referencia_pareto_8D.csv'
if legacy.exists(): sources.append(('NSGAIII_legado',pd.read_csv(legacy)[xs].to_numpy(float)))
X=np.vstack([v for _,v in sources]); labels=np.concatenate([np.repeat(k,len(v)) for k,v in sources])
finite=np.isfinite(X).all(1); feasible=np.linalg.norm(X,axis=1)<=ALPHA+1e-8; X=X[finite&feasible]; labels=labels[finite&feasible]
rounded=np.round(X,10); _,idx=np.unique(rounded,axis=0,return_index=True); idx=np.sort(idx); X=X[idx]; labels=labels[idx]
F=model.predict(X); Fmin=F*signs
idx_nd=NonDominatedSorting(method='efficient_non_dominated_sort').do(Fmin,only_non_dominated_front=True)
Xstar=X[idx_nd]; Fstar=F[idx_nd]; labels_star=labels[idx_nd]
order=np.lexsort(Xstar[:,::-1].T); Xstar=Xstar[order]; Fstar=Fstar[order]; labels_star=labels_star[order]
front=pd.DataFrame(np.column_stack([Xstar,Fstar]),columns=xs+ys); front.insert(0,'source',labels_star)
csv=FINAL/'referencia_pareto_8D_nsga3_moead.csv'; front.to_csv(csv,index=False,encoding='utf-8-sig')
ideal=Fmin[idx_nd].min(0); nadir=Fmin[idx_nd].max(0); amp=np.maximum(nadir-ideal,1e-12); Fn=(Fmin[idx_nd]-ideal)/amp
rows=[]
for name,Xs in sources:
    Fs=model.predict(Xs)*signs; Fsn=(Fs-ideal)/amp; dist=cKDTree(Fn).query(Fsn,k=1)[0]
    rows.append({'source':name,'n_input':len(Xs),'median_distance_to_Pstar':float(np.median(dist)),'p95_distance_to_Pstar':float(np.percentile(dist,95)),'max_distance_to_Pstar':float(dist.max()),'n_retained_as_origin':int((labels_star==name).sum())})
audit=pd.DataFrame(rows); audit.to_csv(FINAL/'referencia_pareto_8D_auditoria_fontes.csv',index=False)
manifest={'csv':str(csv),'sha256':hashlib.sha256(csv.read_bytes()).hexdigest(),'n_candidates':int(sum(len(v) for _,v in sources)),'n_unique_feasible':int(len(X)),'n_final':int(len(front)),'sources':[k for k,_ in sources],'run_config_hash':config_hash,'excel_sha256':cfg['excel_sha256']}
(FINAL/'referencia_pareto_8D_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print(json.dumps(manifest,indent=2)); display(audit); display(front.head())
